← [Overview](00_overview.ipynb)

# Rescaling and denormalisation

[Representation](03_representation.ipynb) chose one profile per cluster. Every rule except `mean`
chose it by **selecting or reshaping** rather than averaging — and only averaging is guaranteed to
keep the cluster's totals. So the representatives now say something slightly untrue about how much
solar and load the series contained.

Rescaling fixes the totals; denormalisation then returns everything to physical units. Together
they are the last thing that happens to a representative before you get it back.

| | |
|---|---|
| **In** | one representative per cluster, plus how many periods each stands for |
| **Inside** | scale the representatives until the weighted totals match the original |
| **Out** | corrected profiles, back in W/m² and MW |

This is the fifth of the [five aspects](00_overview.ipynb) — the one the review lists as needed
*"in the case of non-centroid based clustering algorithms"*.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"

ATTRS = ["solar", "load"]
UNITS = {"solar": "W/m²", "load": "MW"}
N_TIMESTEPS = 4

tiny = pd.read_csv("../../data/tiny.csv", index_col=0, parse_dates=True)
D = pd.read_csv("../../data/tiny_periods.csv", header=[0, 1], index_col=0)

# Pick up exactly where 03 left off: the k=2 Ward partition, represented by medoids.
partition = tsam.aggregate(
    tiny,
    n_clusters=2,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    preserve_column_means=False,  # rescaling off, so we can watch it happen
)
assignments = [int(c) for c in partition.cluster_assignments]
clusters = {
    c: [d for d, a in enumerate(assignments) if a == c]
    for c in sorted(set(assignments))
}
print("k=2 clusters:", clusters)

## 1  What comes in: totals that drifted

Each representative stands in for **every** period in its cluster, so the series it implies is the
representative repeated as many times as the cluster is large. That reconstruction should carry the
same totals as the original. With medoids it does not: the medoid of cluster 0 is day4, and day4 is
not obliged to have an average amount of anything.

In [ ]:
def medoid_index(matrix):
    """03's rule: the member closest to its cluster-mates."""
    dist = np.sqrt(((matrix[:, None, :] - matrix[None, :, :]) ** 2).sum(-1))
    return int(np.argmin(dist.sum(axis=0)))


centres, weights, chosen = [], [], {}
for c, days in clusters.items():
    members = D.loc[days].values
    i = medoid_index(members)
    centres.append(members[i])
    weights.append(len(days))
    chosen[c] = days[i]
centres = np.array(centres)
weights = np.array(weights)

print("cluster -> medoid, occurrence count")
for c, days in clusters.items():
    print(f"  cluster {c}: day{chosen[c]} stands for {len(days)} day(s)")

print("\nDoes the weighted reconstruction carry the original totals? (normalized)")
drift = {}
for a, attr in enumerate(ATTRS):
    block = slice(a * N_TIMESTEPS, (a + 1) * N_TIMESTEPS)
    original = D[attr].values.sum()
    implied = (weights * centres[:, block].sum(axis=1)).sum()
    drift[attr] = (original, implied, original / implied)
    print(
        f"  {attr:6s} original {original:.4f}   representatives imply {implied:.4f}   "
        f"off by {100 * (implied / original - 1):+.1f}%"
    )

The medoids under-supply solar and over-supply load. Left alone, a model built on these periods
would see a sunnier-than-real or hungrier-than-real year.

## 2  Inside: one factor, applied until it sticks

The correction is a multiplicative factor per attribute — the ratio of what there should be to what
the representatives currently imply:

$$
c^{*}_{k,a,t} = c_{k,a,t} \cdot
\frac{\sum_{p}\sum_{t} x_{p,a,t}}
{\sum_{k} \left(\lvert \mathbb{C}_k \rvert \sum_{t} c_{k,a,t}\right)}
\quad \forall\; k, a, t
$$

The numerator is the original total; the denominator is the occurrence-weighted total the
representatives currently produce. If they already match, the factor is 1 and nothing happens.

**But one multiplication is not enough**, and the reason is worth understanding: these values are
**normalized to $[0, 1]$**. Scaling up a profile that already peaks near 1 would push it above the
maximum the attribute ever reached — inventing sunshine that never fell. So tsam **clips** every
scaled value back into range. Clipping removes exactly the increase the factor was counting on, so
the total lands short, and the factor has to be computed and applied **again**. Rescaling is a
loop, not a formula.

In [ ]:
print("The ideal one-shot factor, and what it would do to the ceiling:\n")
for a, attr in enumerate(ATTRS):
    block = slice(a * N_TIMESTEPS, (a + 1) * N_TIMESTEPS)
    original, implied, factor = drift[attr]
    would_breach = int((centres[:, block] * factor > 1.0).sum())
    print(f"  {attr:6s} factor = {original:.4f} / {implied:.4f} = {factor:.4f}")
    print(f"         values pushed above the 1.0 ceiling: {would_breach}\n")

Solar needs to grow by 13.9%, and doing so would push **one** value through the ceiling — day0's
brightest timestep, which sits at exactly 1.0 because it *is* the sunniest moment in the series.
Load needs to shrink, so it breaches nothing.

Below is tsam's actual loop ([`rescale.py`](../background/architecture/pipeline_guide.md)):
rescale, clip, re-measure, repeat — until the total is within `rescale_tolerance` or
`rescale_max_iterations` runs out.

In [ ]:
def rescale_column(values, weights, target, cap=1.0, tolerance=1e-6, max_iterations=20):
    """tsam's rescaling loop for one attribute: scale, clip, re-measure, repeat."""
    values = values.copy()
    total = (weights * values.sum(axis=1)).sum()
    history = []
    iteration = 0
    while abs(target - total) > target * tolerance and iteration < max_iterations:
        values *= target / total
        pinned = int((values > cap).sum())
        values = np.clip(values, 0, cap)
        total = (weights * values.sum(axis=1)).sum()
        iteration += 1
        history.append((iteration, pinned, total, abs(target - total) / target * 100))
    return values, history, iteration < max_iterations


for a, attr in enumerate(ATTRS):
    block = slice(a * N_TIMESTEPS, (a + 1) * N_TIMESTEPS)
    target = D[attr].values.sum()
    _, history, converged = rescale_column(centres[:, block], weights, target)
    print(f"{attr} — target total {target:.4f}")
    for iteration, pinned, total, gap in history[:6]:
        print(
            f"    pass {iteration}: clipped {pinned} value(s), total {total:.4f}, still {gap:.4f}% short"
        )
    print(f"    -> {len(history)} passes, converged={converged}\n")

Load takes a single pass — nothing clips, so the first factor is exact. Solar takes several: each
pass gives back part of what the clip removed, and the gap shrinks geometrically.

tsam reports the outcome per column, so you never have to guess whether it worked:

In [ ]:
rescaled = tsam.aggregate(
    tiny,
    n_clusters=2,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    preserve_column_means=True,  # the default
)

print("tsam's own report (result.accuracy.rescale_deviations):")
print(rescaled.accuracy.rescale_deviations.to_string())

print("\nColumn means — the point of the exercise:")
pd.DataFrame(
    {
        "original": tiny.mean(),
        "without rescaling": partition.reconstructed.mean(),
        "with rescaling": rescaled.reconstructed.mean(),
    }
).round(4)

### When the ceiling wins

Clipping is a hard constraint, and a factor large enough will lose to it. The clearest case is the
`maxoid` representation: it deliberately selects the **most extreme** period of each cluster, so the
representatives are by construction the least average periods available — and the totals they imply
are the furthest from the truth. Ask for a big enough correction and too many values pin against the
ceiling for the loop to ever close the gap.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    extreme_reps = tsam.aggregate(
        tiny,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation="maxoid"),
        preserve_column_means=True,
    )

print("maxoid representatives, rescaling on:")
print(extreme_reps.accuracy.rescale_deviations.to_string())
for w in caught:
    print(f"\nwarning raised:\n  {w.message}")

`converged = False`, and tsam says so rather than quietly returning a series whose solar total is
wrong. The deviation is small here, but the mechanism is not a rounding artefact: it is the
representation and the normalisation bound pulling in opposite directions. **`maxoid` and
`preserve_column_means` want incompatible things** — one insists the representative be an extreme
period, the other insists the fleet of representatives average out correctly. Something has to give,
and the bound is not negotiable.

It is worth knowing which lever to reach for when this warning appears: relax the representation, or
accept the deviation and know its size.

### Extreme clusters are left alone

One exclusion, and [04](04_extreme_periods.ipynb) is the reason. A cluster created to hold a
deliberately extreme period would be *destroyed* by rescaling — the peak would be scaled away, which
is precisely what appending it was meant to prevent. So rescaling skips extreme clusters entirely
and distributes the whole correction across the ordinary ones.

## 3  Denormalisation: back to physical units

Everything so far happened in the normalized $[0, 1]$ space that
[preprocessing](01_preprocessing.ipynb) created. The final step inverts it exactly:

$$
c'^{*}_{k,a,t} = c^{*}_{k,a,t} \left(\max x'_a - \min x'_a\right) + \min x'_a
\quad \forall\; a
$$

One subtlety worth noting: the min and max are the ones measured on the **original** series in
preprocessing, not on the representatives. That is what keeps every typical period on the same
scale as the data it came from.

In [ ]:
def denormalize(profile):
    """The exact inverse of 01's min-max normalization."""
    out = profile.reshape(len(ATTRS), N_TIMESTEPS).copy()
    for a, attr in enumerate(ATTRS):
        low, high = tiny[attr].min(), tiny[attr].max()
        out[a] = out[a] * (high - low) + low
    return out.ravel()


# Rescale cluster 0's medoid by hand, then denormalize it.
final = centres.copy()
for a, attr in enumerate(ATTRS):
    block = slice(a * N_TIMESTEPS, (a + 1) * N_TIMESTEPS)
    final[:, block], _, _ = rescale_column(
        centres[:, block], weights, D[attr].values.sum()
    )

by_hand = pd.DataFrame(
    [denormalize(final[c]) for c in range(len(centres))],
    columns=D.columns,
    index=[f"cluster {c}" for c in range(len(centres))],
)
print("Hand-computed: rescaled, then denormalized to physical units:")
print(by_hand.round(3).to_string())

print("\ntsam's cluster_representatives, for comparison:")
print(rescaled.cluster_representatives.round(3).to_string())

---

**Up next:**

* [Segmentation](06_segmentation.ipynb) — the last transform: merge adjacent timesteps of the
  typical periods into fewer, longer segments

**See also:**

* [Representation](03_representation.ipynb) — non-`mean` representatives are what make rescaling
  necessary in the first place
* [Extreme periods](04_extreme_periods.ipynb) — the clusters rescaling deliberately skips
* [Notation and equations](../../reference/notation.md) — every symbol and formula on one page